# ZarAI — Live Chat with Automatic Validation (Qalb-1.0-8B-Instruct, 4-bit)

> **This version runs Qalb-1.0-8B-Instruct** — a Llama-3.1-8B model that was continually pre-trained on ~1.97 billion Urdu tokens and then instruction-tuned specifically for Urdu — in 4-bit (via bitsandbytes) so it fits on a single T4 GPU. Everything else — matching, scoring, chat UI, and the whole-dataset validation section — is identical to your other notebooks, so results are directly comparable.

**Note on access:** Qalb is a community model (not a Meta-gated repo like Llama-3.1 base), so it typically does **not** require requesting access. The Hugging Face login cell below is kept just in case — if the model loads without it, you can skip logging in entirely.

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas openpyxl sentence-transformers ipywidgets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1

## 2. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** before running this, or
inference will be slow.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## 2b. Hugging Face login (usually not required for this model)

Qalb-1.0-8B-Instruct is typically public. Try skipping this cell first and go straight to loading the model — only come back and run this if you hit a 401/403 error.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Upload your dataset

Click **Choose Files** and select your dataset Excel file.

In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_PATH = list(uploaded.keys())[0]
print("Uploaded:", DATA_PATH)

Saving urdu- agriculture Question Asnwer.xlsx to urdu- agriculture Question Asnwer.xlsx
Uploaded: urdu- agriculture Question Asnwer.xlsx


In [ ]:
import pandas as pd

df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
df.head(3)

Shape: (1000, 13)


,id,question_urdu,question_roman_urdu,question_english,answer_urdu,answer_roman_urdu,answer_english,category,crop,season,region,difficulty_level,source_type
0,1,گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟,Gandum mein peela zang ki bimari kaise pehchan...,How to identify yellow rust disease in wheat?,گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاری...,Gandum mein peela zang ki alamaat: patton par ...,Yellow rust in wheat shows yellow stripes on l...,disease_management,wheat,rabi,Punjab,intermediate,expert_knowledge
1,2,گندم کو پانی کب دیں؟,Gandum ko paani kab dein?,When to irrigate wheat?,گندم کو 4 سنچائی چاہیں: کراؤن روٹ (21 دن)، کشت...,Gandum ko 4 senchai chahiyein: crown root (21 ...,Wheat needs 4 irrigations: crown root (21 days...,irrigation_management,wheat,rabi,Punjab,beginner,agricultural_extension
2,3,گندم کی بہترین اقسام کون سی ہیں؟,Gandum ki behtareen aqsaam konsi hain?,What are the best wheat varieties?,پنجاب میں NARC-2009، فیصل آباد-2008، اور پاکست...,Punjab mein NARC-2009; Faisalabad-2008; aur Pa...,In Punjab; NARC-2009; Faisalabad-2008; and Pak...,variety_selection,wheat,rabi,Punjab,beginner,agricultural_extension


## 4. Load Qalb-1.0-8B-Instruct

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "enstazao/Qalb-1.0-8B-Instruct"

# 4-bit quantization config — the Qalb paper reports its 4-bit version retains ~95% of full
# performance while using ~25% of the memory, so this is a well-validated setup for this model.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Qalb'''s tokenizer config doesn'''t ship a chat_template, even though it'''s a Llama-3.1-8B-Instruct
# continued pre-train under the hood. Its special tokens (<|start_header_id|>, <|eot_id|>, etc.)
# are identical to the base model, so we manually attach the standard Llama-3.1 chat template.
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% set loop_messages = messages %}"
        "{% for message in loop_messages %}"
        "{% set content = '''<|start_header_id|>''' + message['''role'''] + '''<|end_header_id|>\n\n''' "
        "+ message['''content'''] | trim + '''<|eot_id|>''' %}"
        "{% if loop.index0 == 0 %}{% set content = '''<|begin_of_text|>''' + content %}{% endif %}"
        "{{ content }}"
        "{% endfor %}"
        "{% if add_generation_prompt %}"
        "{{ '''<|start_header_id|>assistant<|end_header_id|>\n\n''' }}"
        "{% endif %}"
    )
    print("Chat template was missing — manually attached the Llama-3.1-Instruct template.")
else:
    print("Chat template already present, no patch needed.")

print("Model loaded (4-bit):", MODEL_NAME)
print("GPU memory allocated: {:.2f} GB".format(torch.cuda.memory_allocated() / 1e9))

config.json:   0%|          | 0.00/918 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Chat template was missing — manually attached the Llama-3.1-Instruct template.
Model loaded (4-bit): enstazao/Qalb-1.0-8B-Instruct
GPU memory allocated: 5.71 GB


## 5. Define the inference function

In [ ]:
def ask_model(question, system_prompt="You are a helpful agricultural advisor for farmers in Pakistan. Answer clearly and practically."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True
    )
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# quick sanity check
print(ask_model("گندم کو پانی کب دیں؟"))

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔

گندم کو پانی کب دیں؟assistant

گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔
گندم کو پانی کب دیں؟assistant

گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔
گندم کو پانی کب دیں؟assistant

گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔
گندم کو پانی کب دیں؟assistant

گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔
گندم کو پانی کب دیں؟assistant

گندم کو پانی دیں جب اس کی پتے بڑھنا شروع ہوں۔
گندم کو پ


## 6. Load the embedding model and build a searchable index of your dataset

Embeds every question in your dataset (all three language columns) once, so
that any question you type later can be matched against it instantly.

This may take a minute or two the first time (encoding ~3x your row count).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print("Embedding model loaded:", EMBED_MODEL_NAME)

def build_reference_index(df):
    records = []
    lang_cols = [
        ("urdu", "question_urdu", "answer_urdu"),
        ("roman_urdu", "question_roman_urdu", "answer_roman_urdu"),
        ("english", "question_english", "answer_english"),
    ]
    for _, row in df.iterrows():
        for lang, qcol, acol in lang_cols:
            q = row.get(qcol)
            if pd.notna(q) and str(q).strip():
                records.append({
                    "id": row.get("id"),
                    "category": row.get("category"),
                    "crop": row.get("crop"),
                    "language": lang,
                    "question": str(q),
                    "answer": row.get(acol),
                })
    ref_df = pd.DataFrame(records)
    embeddings = embed_model.encode(
        ref_df["question"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=64,
    )
    ref_df["embedding"] = list(embeddings)
    return ref_df

reference_index = build_reference_index(df)
print(f"Indexed {len(reference_index)} questions across all languages.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Indexed 3000 questions across all languages.


## 7. Matching + scoring functions

In [ ]:
def find_best_match(user_question):
    q_emb = embed_model.encode([user_question], normalize_embeddings=True)[0]
    sims = np.array([float(np.dot(q_emb, e)) for e in reference_index["embedding"]])
    best_idx = int(sims.argmax())
    best_row = reference_index.iloc[best_idx]
    return best_row, sims[best_idx] * 100

def semantic_similarity(text_a, text_b):
    emb = embed_model.encode([str(text_a), str(text_b)], normalize_embeddings=True)
    return float(np.dot(emb[0], emb[1])) * 100

# Below this match confidence, we don't trust the "reference answer" enough to score against it
MATCH_CONFIDENCE_THRESHOLD = 35

## 8. Live chat interface

Type a question, hit **Send** (or press Enter), and you'll see:

- Your question
- The model's live answer
- A validation panel: which dataset question it matched, the reference
  answer, and a semantic accuracy score (color-coded: green = strong match,
  amber = partial, red = weak/no reliable match)

Use **Export to Excel** any time to download everything asked so far.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

chat_log = []

def score_color(score):
    if score is None:
        return "#888888"
    if score >= 70:
        return "#2e7d32"
    elif score >= 40:
        return "#f9a825"
    else:
        return "#c62828"

output_area = widgets.Output(
    layout=widgets.Layout(border="1px solid #ccc", height="480px", overflow_y="auto", padding="10px")
)
text_input = widgets.Text(
    placeholder="Type your agricultural question here (Urdu, Roman Urdu, or English)...",
    layout=widgets.Layout(width="78%")
)
send_button = widgets.Button(description="Send", button_style="primary")
export_button = widgets.Button(description="Export to Excel", button_style="info")

def handle_send(_):
    question = text_input.value.strip()
    if not question:
        return
    text_input.value = ""

    with output_area:
        display(HTML(
            f"<div style=\'text-align:right; margin:8px 0;\'>"
            f"<span style=\'background:#2196F3; color:white; padding:8px 12px; "
            f"border-radius:12px; display:inline-block; max-width:75%;\'>{question}</span></div>"
        ))

    model_answer = ask_model(question)

    with output_area:
        display(HTML(
            f"<div style=\'text-align:left; margin:8px 0;\'>"
            f"<span style=\'background:#f1f1f1; padding:8px 12px; "
            f"border-radius:12px; display:inline-block; max-width:75%;\'>{model_answer}</span></div>"
        ))

    match_row, match_confidence = find_best_match(question)
    reference_answer = match_row["answer"]

    if pd.notna(reference_answer) and match_confidence >= MATCH_CONFIDENCE_THRESHOLD:
        accuracy_score = semantic_similarity(model_answer, reference_answer)
        color = score_color(accuracy_score)
        val_html = f"""
        <div style=\'margin:4px 0 16px 0; padding:10px; background:#fafafa;
                     border-left:4px solid {color}; font-size:13px;\'>
        <b>Validation</b><br>
        Matched dataset question ({match_row['language']}, match confidence {match_confidence:.1f}/100):<br>
        <i>{match_row['question']}</i><br>
        Reference answer: {reference_answer}<br>
        <span style=\'color:{color}; font-weight:bold;\'>Semantic accuracy score: {accuracy_score:.1f}/100</span>
        </div>
        """
    else:
        accuracy_score = None
        val_html = f"""
        <div style=\'margin:4px 0 16px 0; padding:10px; background:#fafafa;
                     border-left:4px solid #888888; font-size:13px;\'>
        <b>Validation</b><br>
        No close match found in the dataset (closest match confidence {match_confidence:.1f}/100) —
        can't validate against a known ground truth for this question.
        </div>
        """

    with output_area:
        display(HTML(val_html))

    chat_log.append({
        "question": question,
        "model_answer": model_answer,
        "matched_question": match_row["question"],
        "matched_language": match_row["language"],
        "match_confidence": round(match_confidence, 1),
        "reference_answer": reference_answer,
        "semantic_accuracy_score": accuracy_score,
    })

def handle_export(_):
    if not chat_log:
        print("No chat history yet — ask something first.")
        return
    out_path = "zarai_qalb_8b_chat_session_results.xlsx"
    pd.DataFrame(chat_log).to_excel(out_path, index=False)
    files.download(out_path)
    print("Exported:", out_path)

send_button.on_click(handle_send)
text_input.on_submit(handle_send)  # Enter key also sends
export_button.on_click(handle_export)

display(widgets.VBox([
    widgets.HTML("<h3>ZarAI Chat — Ask an agricultural question</h3>"),
    output_area,
    widgets.HBox([text_input, send_button]),
    export_button
]))

## 9. Whole-dataset validation (batch, no typing required)

This section runs the **same SBERT-based validation** used in the live chat above,
but automatically across every question in your dataset (all three languages),
instead of one-at-a-time typing.

For each row it will:
1. Send the question (Urdu / Roman Urdu / English) to Qwen2.5 via `ask_model()` — identical function to the chat above
2. Score the model's answer against that row's own ground-truth answer using `semantic_similarity()` — identical scoring function
3. Also run `find_best_match()` as a sanity check (should match itself with high confidence — a low self-match confidence usually flags a near-duplicate question elsewhere in the dataset, not a bug)
4. Log everything, then export a multi-sheet Excel report with overall + breakdown stats

**Note:** Generating ~3,000 model answers (1,000 rows × 3 languages) on a single T4 GPU will take a while
(roughly 3–6+ seconds per answer (similar to the 7B Qwen run — an 8B model in 4-bit is a comparable amount of compute per token) depending on `max_new_tokens`, so plan for 1–2+ hours for the full set).
Use `SAMPLE_SIZE` below to run a quick stratified subset first (e.g. 100–150 rows) to sanity-check everything,
then set it to `None` to run the full 1,000-row dataset overnight or across a longer session.

In [ ]:
import time
from tqdm.auto import tqdm

# ---- Config ----
SAMPLE_SIZE = 100          # None = run the FULL dataset (~3,000 questions); int = stratified sample of N rows by category
LANGUAGES_TO_RUN = ["urdu", "roman_urdu", "english"]   # drop any language here to skip it
CHECKPOINT_EVERY = 25       # rows (per language) between autosaves, so a Colab disconnect doesn't lose progress
CHECKPOINT_PATH = "zarai_qalb_8b_full_validation_checkpoint.xlsx"  # separate filename from other model runs

lang_cols = {
    "urdu": ("question_urdu", "answer_urdu"),
    "roman_urdu": ("question_roman_urdu", "answer_roman_urdu"),
    "english": ("question_english", "answer_english"),
}

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df):
    # Stratified sample by category — uses groupby().sample(), NOT groupby().apply(lambda x: x.sample(...)),
    # since the .apply() version silently drops the grouping column under pandas 3.0+.
    n_per_group = max(1, SAMPLE_SIZE // df["category"].nunique())
    eval_df = (
        df.groupby("category", group_keys=False)
          .sample(n=n_per_group, random_state=42, replace=False)
          .reset_index(drop=True)
    )
    # Trim/pad to get close to SAMPLE_SIZE exactly
    eval_df = eval_df.sample(n=min(SAMPLE_SIZE, len(eval_df)), random_state=42).reset_index(drop=True)
else:
    eval_df = df.reset_index(drop=True)

print(f"Rows to evaluate: {len(eval_df)}  |  Languages: {LANGUAGES_TO_RUN}  "
      f"|  Total question/answer pairs: {len(eval_df) * len(LANGUAGES_TO_RUN)}")

Rows to evaluate: 84  |  Languages: ['urdu', 'roman_urdu', 'english']  |  Total question/answer pairs: 252


In [ ]:
full_results = []
start_time = time.time()
processed = 0

for lang in LANGUAGES_TO_RUN:
    qcol, acol = lang_cols[lang]
    for row_pos, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Evaluating ({lang})")):
        question = row.get(qcol)
        reference_answer = row.get(acol)

        if pd.isna(question) or not str(question).strip():
            continue  # skip rows missing this language's question

        model_answer = ask_model(str(question))

        if pd.notna(reference_answer) and str(reference_answer).strip():
            accuracy_score = semantic_similarity(model_answer, reference_answer)
        else:
            accuracy_score = None

        # Self-match sanity check: is this question uniquely findable in the reference index?
        match_row, match_confidence = find_best_match(str(question))

        full_results.append({
            "id": row.get("id"),
            "language": lang,
            "question": question,
            "model_answer": model_answer,
            "reference_answer": reference_answer,
            "semantic_accuracy_score": accuracy_score,
            "self_match_confidence": round(match_confidence, 1),
            "category": row.get("category"),
            "crop": row.get("crop"),
            "season": row.get("season"),
            "region": row.get("region"),
            "difficulty_level": row.get("difficulty_level"),
            "source_type": row.get("source_type"),
        })

        processed += 1
        if processed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(full_results).to_excel(CHECKPOINT_PATH, index=False)

elapsed = time.time() - start_time
print(f"Done. Evaluated {len(full_results)} question/answer pairs in {elapsed/60:.1f} minutes.")

results_df = pd.DataFrame(full_results)
results_df.head()

Evaluating (urdu):   0%|          | 0/84 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

Evaluating (roman_urdu):   0%|          | 0/84 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

Evaluating (english):   0%|          | 0/84 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

Done. Evaluated 252 question/answer pairs in 96.0 minutes.


,id,language,question,model_answer,reference_answer,semantic_accuracy_score,self_match_confidence,category,crop,season,region,difficulty_level,source_type
0,293,urdu,گندم میں نئی مزاحم اقسام کی فہرست؟,گندم میں نئی مزاحم اقسام کی فہرست یہ ہے: \n1. ...,فیصل آباد-2008، پاکستان-2013، نارک-2009، اور ن...,55.090690,100.0,variety_selection,wheat,rabi,Punjab,intermediate,agricultural_extension
1,100,urdu,نوجوانوں کے لیے زراعت میں روزگار کے مواقع؟,پاکستان میں زراعت میں روزگار کے مواقع کافی ہیں...,ایگری بزنس، فوڈ پروسیسنگ، ڈرون اسپرے، ورٹیکل ف...,43.151754,100.0,agricultural_career,general,all_seasons,All Pakistan,intermediate,agricultural_extension
2,165,urdu,گندم کی دیر سے بوائی کے نقصانات؟,گندم کی دیر سے بوائی کے نقصانات میں پودے کی نش...,پیداوار 20-30% کم ہو جاتی ہے۔ جڑی بوٹیاں اور ب...,71.709090,100.0,planting_calendar,wheat,rabi,All Pakistan,intermediate,agricultural_extension
3,718,urdu,potato میں پاؤڈر کی بیماری کا علاج کیا ہے؟,پودر کی بیماری کا علاج کیسے کیا جائے اس کے لئے...,پاؤڈر کے لیے فنگی سائیڈ کا اسپرے کریں۔,52.344131,100.0,disease_management,potato,all_seasons,All Pakistan,intermediate,expert_knowledge
4,180,urdu,بلوچستان میں زیتون کی کاشت کے فوائد؟,بلوچستان میں زیتون کی کاشت کے فوائد میں زیتون ...,پہاڑی علاقوں میں اچھی پیداوار۔ تیل کی برآمدات ...,46.040320,100.0,crop_geography,olive,perennial,Balochistan/KPK,intermediate,agricultural_extension


## 10. Aggregate stats + export full report

In [ ]:
def summarize(group_col):
    valid = results_df.dropna(subset=["semantic_accuracy_score"])
    return (
        valid.groupby(group_col)["semantic_accuracy_score"]
             .agg(["count", "mean", "median", "std", "min", "max"])
             .round(1)
             .sort_values("mean", ascending=False)
    )

overall = pd.DataFrame({
    "metric": ["total_evaluated", "scored_pairs", "mean_score", "median_score", "std_score"],
    "value": [
        len(results_df),
        results_df["semantic_accuracy_score"].notna().sum(),
        round(results_df["semantic_accuracy_score"].mean(), 1),
        round(results_df["semantic_accuracy_score"].median(), 1),
        round(results_df["semantic_accuracy_score"].std(), 1),
    ],
})

by_language = summarize("language")
by_category = summarize("category")
by_crop = summarize("crop")
by_difficulty = summarize("difficulty_level")
by_source_type = summarize("source_type")

# Flag rows worth a manual look: weak self-match (possible near-duplicate question) or low accuracy
low_confidence_flags = results_df[results_df["self_match_confidence"] < 90]
low_accuracy_flags = results_df[results_df["semantic_accuracy_score"] < 40]

print("Overall:")
print(overall.to_string(index=False))
print("\nBy language:")
print(by_language)
print(f"\nRows with weak self-match (<90/100): {len(low_confidence_flags)}")
print(f"Rows with low semantic accuracy (<40/100): {len(low_accuracy_flags)}")

Overall:
         metric  value
total_evaluated  252.0
   scored_pairs  252.0
     mean_score   50.2
   median_score   51.5
      std_score   17.2

By language:
            count  mean  median   std   min   max
language                                         
english        84  57.6    55.5  12.5  23.7  83.8
urdu           84  52.4    52.5  17.8  11.8  93.0
roman_urdu     84  40.6    38.9  16.3  12.8  77.0

Rows with weak self-match (<90/100): 0
Rows with low semantic accuracy (<40/100): 72


In [ ]:
FULL_REPORT_PATH = "zarai_qalb_8b_full_dataset_validation.xlsx"  # separate filename from other model runs

with pd.ExcelWriter(FULL_REPORT_PATH, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="all_results", index=False)
    overall.to_excel(writer, sheet_name="overall_summary", index=False)
    by_language.to_excel(writer, sheet_name="by_language")
    by_category.to_excel(writer, sheet_name="by_category")
    by_crop.to_excel(writer, sheet_name="by_crop")
    by_difficulty.to_excel(writer, sheet_name="by_difficulty")
    by_source_type.to_excel(writer, sheet_name="by_source_type")
    low_confidence_flags.to_excel(writer, sheet_name="flag_weak_self_match", index=False)
    low_accuracy_flags.to_excel(writer, sheet_name="flag_low_accuracy", index=False)

files.download(FULL_REPORT_PATH)
print("Exported:", FULL_REPORT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Exported: zarai_qalb_8b_full_dataset_validation.xlsx
